In [1]:
# Загрузка сырых данных

import pandas as pd

df_sales = pd.read_csv(
    "sales_dataset_hakaton_train.csv",
    parse_dates=["date"]
)

df_stock = pd.read_csv(
    "stock_dataset_hakaton_train.csv",
    parse_dates=["date"]
)

df_sales.head()

,date,composite_division,cities_name,warehouse_name,category_lvl_1,category_lvl_2,category_lvl_3,article,characteristic,quantity,sales_sum,discount_sum
0,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_0,category_lvl_2_0,NaN,char_article_0,characteristic_0,2.0,7.429034,5.459302
1,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_1,category_lvl_2_1,NaN,char_article_1,characteristic_0,1.0,9.613652,7.798864
2,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_2,category_lvl_2_2,NaN,char_article_2,characteristic_0,1.0,9.237081,7.258904
3,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_3,category_lvl_2_3,NaN,char_article_3,characteristic_1,1.0,9.066837,7.094100
4,2024-01-01,composite_division_0,cities_name_1,warehouse_name_1,category_lvl_1_2,category_lvl_2_2,NaN,char_article_4,characteristic_0,1.0,9.237083,7.258905


In [2]:
df_stock.head()

,date,warehouse_name,article,stock
0,2023-12-13,warehouse_name_0,char_article_119,1
1,2023-12-13,warehouse_name_3,char_article_62,135
2,2023-12-13,warehouse_name_3,char_article_268,3
3,2023-12-13,warehouse_name_7,char_article_258,1
4,2023-12-13,warehouse_name_3,char_article_14,19


In [ ]:
keys = ["date", "warehouse_name", "article"]


# Агрегация остатков
df_stock_clean = df_stock.groupby(
    keys, as_index=False
).agg(
    stock=("stock", "sum")
)

# Проверка
print("Строк после агрегации:", len(df_stock_clean))
print("Дубликатов осталось:", df_stock_clean.duplicated(keys).sum())

Строк после агрегации: 4884198
Дубликатов осталось: 0


In [4]:
#границы дат для всей истории
min_date = min(df_stock_clean['date'].min(), df_sales['date'].min())
max_date = max(df_stock_clean['date'].max(), df_sales['date'].max())

# непрерывный календарь (каждый день)
all_dates = pd.date_range(min_date, max_date, freq='D')

# уникальные списки для сетки
warehouses = df_sales['warehouse_name'].unique()
articles = df_sales['article'].unique()

In [5]:
keys = ["date", "warehouse_name", "article"]

# полная сетка всех комбинаций (склад × товар × дата)
full_grid = pd.MultiIndex.from_product(
    [warehouses, articles, all_dates],
    names=['warehouse_name', 'article', 'date']
).to_frame(index=False)

# Присоединение остатков к сетке
df_stock_daily = full_grid.merge(
    df_stock_clean,
    on=keys,
    how='left'
)


In [6]:
# Сортировка по датам
df_stock_daily = df_stock_daily.sort_values(['warehouse_name', 'article', 'date'])
# флаг отсутствия данных
df_stock_daily['stock_was_nan'] = df_stock_daily['stock'].isna().astype(int)
# Заполнение остатков вперед (тянем последнее известное значение)
df_stock_daily['stock'] = (
    df_stock_daily.groupby(['warehouse_name', 'article'])['stock']
    .ffill()
)

print("Размер ежедневного датасета остатков:", df_stock_daily.shape)
print("Пропусков в stock:", df_stock_daily['stock'].isna().sum())

Размер ежедневного датасета остатков: (8455018, 5)
Пропусков в stock: 3547699


In [7]:
# Подсчет количества строк по формуле
expected_rows = 11 * 799 * len(all_dates) # all_dates создавали при построении сетки
print(f"Ожидалось строк: {expected_rows}")
print(f"Фактически:      {len(df_stock_daily)}")

Ожидалось строк: 8455018
Фактически:      8455018


In [8]:
#Проверка отсутствия дубликатов по ключам
dups = df_stock_daily.duplicated(subset=['date', 'warehouse_name', 'article']).sum()
print(f"Дубликатов: {dups} (должно быть 0)")

Дубликатов: 0 (должно быть 0)


In [9]:
# проверка пропусков
print(f"Пропусков (NaN) в stock: {df_stock_daily['stock'].isna().sum()}")


Пропусков (NaN) в stock: 3547699


In [10]:
# Выбор одного случайный товара для проверки непрерывности календаря
sample = df_stock_daily[
    (df_stock_daily['warehouse_name'] == 'warehouse_name_0') & 
    (df_stock_daily['article'] == 'char_article_0')
].sort_values('date')

In [11]:
# Подсчет разницы между соседними датами
date_diffs = sample['date'].diff().dropna().dt.days
print(f"Максимальный разрыв между датами: {date_diffs.max()} дней (должно быть 1)")

Максимальный разрыв между датами: 1 дней (должно быть 1)


In [12]:
# 1. Проверка количества отрицательных остатков
neg_stock = (df_stock_daily['stock'] < 0).sum()
print(f"Отрицательных остатков (stock < 0): {neg_stock}")



Отрицательных остатков (stock < 0): 31


In [13]:

keys = ["date", "warehouse_name", "article"]
cat_cols = ["cities_name", "category_lvl_1", "category_lvl_2"]

# Заменяем null в скидках на 0
df_sales["discount_sum"] = df_sales["discount_sum"].fillna(0)

# 2. Строим чистый справочник "артикул -> категория" (без пустой category_lvl_3)
df_items_directory = df_sales[["article"] + cat_cols].dropna().drop_duplicates(subset=["article"])

# Проверяем на отрицательные продажи
neg = (df_sales["quantity"] < 0).sum()
print(f"Строк с отрицательным quantity: {neg}")

Строк с отрицательным quantity: 0


In [14]:
# Суммирование продаж по департаментам
df_sales_clean = (
    df_sales
    .groupby(keys, as_index=False)
    .agg(
        quantity=("quantity", "sum")
    )
)

In [15]:
# Проверка
print("Дубликатов после groupby:", df_sales_clean.duplicated(keys).sum())  # должно быть 0
print("Пропусков в quantity:", df_sales_clean["quantity"].isna().sum())   # должно быть 0
print("Мин. quantity:", df_sales_clean["quantity"].min())                  # должно быть >= 0

Дубликатов после groupby: 0
Пропусков в quantity: 0
Мин. quantity: 1.0


In [ ]:
# проверка

print("Проверка справочника категорий (df_items_directory):")
for col in ['cities_name', 'category_lvl_1', 'category_lvl_2']:
    if col in df_items_directory.columns:
        print(f"   - Пропусков в {col}: {df_items_directory[col].isna().sum()}")

print(f"   - Всего уникальных товаров в справочнике: {df_items_directory['article'].nunique()}")

print(f"\nИтоги по агрегированным продажам (df_sales_clean):")
print(f"   - Строк в очищенном sales: {len(df_sales_clean)}")
print(f"   - Уникальных дат: {df_sales_clean['date'].nunique()}")
print(f"   - Уникальных складов: {df_sales_clean['warehouse_name'].nunique()}")
print(f"   - Уникальных товаров с продажами: {df_sales_clean['article'].nunique()}")


1. Проверка справочника категорий (df_items_directory):
   - Пропусков в cities_name: 0
   - Пропусков в category_lvl_1: 0
   - Пропусков в category_lvl_2: 0
   - Всего уникальных товаров в справочнике: 799

2. Итоги по агрегированным продажам (df_sales_clean):
   - Строк в очищенном sales: 970677
   - Уникальных дат: 943
   - Уникальных складов: 11
   - Уникальных товаров с продажами: 799


In [19]:
keys = ["date", "warehouse_name", "article"]
cat_cols = ["cities_name", "category_lvl_1", "category_lvl_2"]

# Присоединяем продажи к сетке стоков
df_final = df_stock_daily.merge(
    df_sales_clean,
    on=keys,
    how="left" # left, потому что df_stock_daily — это полный календарь
)

# Заполняем продажи нулями (для дней, когда товар не продавался)
df_final["quantity"] = df_final["quantity"].fillna(0)
df_final = df_final.merge(df_items_directory, on="article", how="left")

# Заполняем категории (для товаров, которые есть на складе, но ни разу не продавались)
for col in cat_cols:
    df_final[col] = df_final[col].fillna("unknown").astype("category")

In [20]:

print(f"Размер таблицы: {df_final.shape}")
print(f"Пропусков в quantity: {df_final['quantity'].isna().sum()} (должно быть 0)")
print(f"Пропусков в stock: {df_final['stock'].isna().sum()} ")
print(f"Дубликатов по ключам: {df_final.duplicated(subset=keys).sum()} (должно быть 0)")

Размер таблицы: (8455018, 9)
Пропусков в quantity: 0 (должно быть 0)
Пропусков в stock: 3547699 
Дубликатов по ключам: 0 (должно быть 0)


In [21]:
# Считаем количество уникальных дней в истории
n_dates = df_final['date'].nunique()
n_warehouses = df_final['warehouse_name'].nunique()
n_articles = df_final['article'].nunique()

# Считаем ожидаемое количество строк
expected_rows = n_warehouses * n_articles * n_dates

print(f"Уникальных складов: {n_warehouses} (по ТЗ должно быть 11)")
print(f"Уникальных артикулов: {n_articles} (по ТЗ должно быть 799)")
print(f"Уникальных дней истории: {n_dates}")

print(f"\nОжидаемое количество строк (11 * 799 * {n_dates}): {expected_rows}")
print(f"Фактическое количество строк: {len(df_final)}")

if len(df_final) == expected_rows:
    print("\nКоличество строк совпадает. Сетка полная, дубликатов и пропусков нет.")
else:
    print(f"\nРазница: {expected_rows - len(df_final)} строк. Где-то потерялись данные.")

Уникальных складов: 11 (по ТЗ должно быть 11)
Уникальных артикулов: 799 (по ТЗ должно быть 799)
Уникальных дней истории: 962

Ожидаемое количество строк (11 * 799 * 962): 8455018
Фактическое количество строк: 8455018

Количество строк совпадает. Сетка полная, дубликатов и пропусков нет.


In [22]:
# Проверка непрерывности дат
dates = pd.Series(sorted(df_final['date'].unique()))
gaps = dates.diff().dt.days.dropna()

print(f"Максимальный разрыв между датами: {gaps.max()} день (должен быть 1)")
print(f"Минимальная дата: {dates.min().date()}")
print(f"Максимальная дата: {dates.max().date()}")

Максимальный разрыв между датами: 1.0 день (должен быть 1)
Минимальная дата: 2023-12-13
Максимальная дата: 2026-07-31


In [24]:
import os
import pandas as pd

df_final['date'] = pd.to_datetime(df_final['date']).dt.date

save_path = r"C:\Users\evkho\OneDrive\Рабочий стол\hakaton_datasets\base_grid.csv"

df_final.to_csv(save_path, index=False)

print(f"✅ Файл успешно сохранен по пути: {save_path}")

✅ Файл успешно сохранен по пути: C:\Users\evkho\OneDrive\Рабочий стол\hakaton_datasets\base_grid.csv
